T1 measurement experiment in quantum computing determines how long a qubit stays in it's excited state before losing energy and returning to the ground state.

T1 Measurement Experiment simulates a standard qubit T1 (energy relaxation) experiment:

1. prepare the qubit in |1>.

2. let it idle for variable delay time t.

3. Measure the population remaining in |1>.

4. Repeat over a range of delays, fit P1(t) = A*exp(-t/T1) + C.


Since this project is a simulation (not a real quantum hardware experiment), we cannot directly measure the physical T1 of a real qubit. Instead, we assume a known value of T1 (called true_T1) within the expected range (e.g. 50–200 μs). We then use the Lindblad master equation to simulate how the qubit relaxes over different delay times and obtain the excited-state population. Finally, by fitting the population-vs-delay data to an exponential decay curve, we estimate the T1 value. If the simulation and fitting are correct, the estimated T1 should closely match the chosen true_T1.

as qutip is not inbuilt in colab

In [1]:
%pip install qutip

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 28.5/28.5 MB 32.1 MB/s eta 0:00:00


In [2]:
import numpy as np
import qutip as qt
from scipy.optimize import curve_fit

In [3]:
def validate_T1(T1):
  if T1<=0:
    raise ValueError("T1 must be positive")

In [4]:
def validate_T1_T2(T1, T2):
  if T2<=0:
    raise ValueError("T2 must be positive")
  if T2> 2*T1+1e-12:
    raise ValueError("T2 must be <= 2T1")

Validate the array of idle-time delays used in the experiment


Errors that can happen: if delays is empty or has fewer than 3 points (can't fit 3 free parameters), contains negative values or is not strictly increasing



In [5]:
def validate_delays(delays):
  delays = np.asarray(delays, dtype=float)
  #converts the given delays to numpy array form
  if delays.size < 3:
    raise ValueError(f"Need atleast 3 delay points to fit A, T1, C; got {delays.size}")
  if np.any(delays<0):
    raise ValueError("delays must be non-negative")
  if not np.all(np.diff(delays) > 0):
    raise ValueError("delays must be strictly increasing.")
  return delays

n_shots specifies how many times the experiment is repeated for each delay time.

the below given function checks wether the n_shot is valid or not.

In [6]:
def validate_shots(n_shots):
  """validate the number of single shot repetitions per delay point."""
  if n_shots is not None:
    if n_shots <= 0:
      raise ValueError("n_shots must be positive")
    if not float(n_shots).is_integer():
      raise ValueError("n_shots must be an integer")


Now let us build the T1 (energy relaxation) collapse operator for this experiment:

L_relax = sqrt(1/T1)*sigma_minus,
where sigma_minus = |0><1| lowers the qubit from |1> to |0>, modeling spontaneous energy decay at rate gamma1 = 1/T1

In [7]:
def relaxation_operator(T1):
  """Parameters
  T1 : Energy relaxation time must be positive"""
  validate_T1(T1)
  #checks wether T1 is valid or not
  gamma1 = 1.0/T1
  #gamma1 is relaxarion rate
  return np.sqrt(gamma1)*qt.sigmap()
  #here in traditional quantum mechanics textbooks basis is ordered by spin along z-axis so sigmam() refers to [[0,0], [1,0]]
  #but in qutip this state [[0,0],[1,0]] is represented by sigmap()
  #returns L_relax

build an "optional" background pure dephasing operator

In [8]:
def dephasing_operator(T1, T2):
  """ L_dephase = sqrt(gamma_phi/2) * sigma_z, gamma_phi = 1/T2 - 1/(2*T1)
  Included purely so this experiment can test that background dephasing is not corrupt a T1 fit.

  Parameters:
  T1: Relaxation time
  T2: Dephasing time"""
  validate_T1_T2(T1, T2)
  gamma_phi = 1.0/T2 - 1.0/(2*T1)
  return np.sqrt(gamma_phi/2)*qt.sigmaz()

In [9]:
def build_hamiltonian(detuning=0.0):
  """A T1 experiment has no coherent driven during the idle delay, so detuning defaults to 0 (zero hamiltonian, pure decay).
  The detuning parameter is kept for flexibility, allowing the simulation to test the effect of accidental (stray) detuning during the idle period.
  such detuningshould not affect the extracted T1 value because it only changes the qubit's phase and does not alter the excited-state population"""
  if detuning == 0:
    return 0*qt.qeye(2)
  return 0.5*detuning*qt.sigmaz()

assemble the collapse operators for this T1 experiment by calling the local relaxation_operator

In [10]:
def build_t1_collapse_operators(T1, background_T2=None):
  """Parameters
  T1 : Energy relaxation time must be positive
  background_T2 : background dephasing time"""
  validate_T1(T1)
  c_ops = [relaxation_operator(T1)]
  if background_T2 is not None:
    L_dephase = dephasing_operator(T1, background_T2)
    if L_dephase is not None:
      c_ops.append(L_dephase)
      #adds dephasing operator to the list if it is not empty
  return c_ops

now let us simulate T1 measurement

In [11]:
def run_t1_experiment(T1, delays, background_T2=None, detuning=0.0, n_shots=None, seed=None):
  """Parameters
  T1: energy relaxation time
  delays: idle times at which population is measured. (must be sorted)
  background_T2: optional background dephasing time
  detuning: optional stray detuning during the idle period
  n_shots: ow many times the experiment is repeated for each delay time.
  seed: for reproducable simulated data
  Returns
  delays: delay of times
  pop_ideal : exact |1> population from mesolve
  pop_measured : population used for fitting (== pop_ideal if n_shots is None, otherwise the noisy estimate)
  n_shots: the n_shots value used"""
  delays = validate_delays(delays)
  validate_shots(n_shots)
  c_ops = build_t1_collapse_operators(T1, background_T2=background_T2)
  #creates lindblad collapse operators
  H = build_hamiltonian(detuning=detuning)
  rho0 = qt.ket2dm(qt.basis(2, 1))
   #prepares |1>
  num_op = qt.num(2)
  #projector onto |1> i.e; |1><1|
  # Diagnostic prints before mesolve
  print(f"c_ops before mesolve: {c_ops}")
  print(f"rho0 before mesolve:\n{rho0}")
  result = qt.mesolve(
                      H,
                      rho0,
                      delays,
                      c_ops=c_ops,
                      e_ops=[num_op]
                      )
  print(f"mesolve result.expect[0]: ({result.expect[0]})")
  #Diagonistic print to check mesolve output
  pop_ideal = np.clip(np.real(result.expect[0]),0.0, 1.0)
  #extracts probabilities
  if n_shots is not None:
    rng = np.random.default_rng(seed)
    #creates a random generator
    #populations must have non-zero variance
    counts = rng.binomial(n_shots, pop_ideal)
    #applies binomial distribution to find |1> state counts
    pop_measured = counts/n_shots
  else:
    pop_measured = pop_ideal.copy()
  return  {
      "delays": delays,
      "pop_ideal": pop_ideal,
      "pop_measured": pop_measured,
      "n_shots": n_shots
  }

Here pop_ideal is population when n_shots is none while there will be a very very slight variation in population when n_shots is not none and that population is indicated by pop_measured.

n_shots is None --> infinite number of measurements are taken to calculate probability

Exponential decay model: P1(t) = A*exp(-t/T1) + C

In [12]:
def _t1_decay_model(t, A, T1, C):
  return A*np.exp(-t/T1) + C

Fit measured T1-decay data to P1(t) = A*exp(-t/T1) + C using non-linear least squares (robust to noise and readout offset, unlike a log-linear fit which breaks down for noisy or offset-containing data and handle zero/negative points).

In [13]:
def fit_T1(delays, populations, p0=None):
  """Parameters
  delays : idle times at which population is measured
  populations : excited state probabilities
  p0 : Initial guess for (A, T1, C) --> (A0, T1_0, C0)

  Returns
  dict with keys: 'A', 'T1', 'C', 'A_err', 'T1_err', 'C_err', 'pcov' """
  delays = np.asarray(delays, dtype=float)
  populations = np.asarray(populations, dtype=float)
  #converts into numpy arrays
  if delays.shape != populations.shape:
    raise ValueError("delays and populations must have the same shape")
  if delays.size < 3:
    raise ValueError("Need atleast 3 delay points to fit A, T1, C")
  if np.ptp(populations) < 1e-8:
    raise ValueError("populations must have non-zero variance")
    #np.ptp checks Maximum - Minimum value
  if p0 is None:
    A0 = populations[0] - populations[-1]
    C0 = populations[-1]
    half_life_guess = delays[-1]/3.0 if delays[-1] >0 else 1.0
    p0 = (A0 if A0!= 0 else 1.0, max(half_life_guess, 1e-6), C0)
  #if user doesn't give initial values estimate them
  try :
    popt, pcov = curve_fit(_t1_decay_model, delays, populations, p0=p0, bounds=([-2.0, 1e-9, -1.0], [2.0, np.inf, 1.0]), maxfev=10000)
    #Fits the T1 decay model to the measured data using bounded nonlinear least-squares optimization
    #maxfev restricts upto 10000 iterations
  except RuntimeError as e:
    raise ValueError("curve_fit failed") from e
  #throws a RunTimeError if curve fit is not converged
  A, T1_fit, C = popt
  perr = np.sqrt(np.diag(pcov))
  #computes the standard errors of the fitted parameters from the covariance matrix
  return {
      'A': A,
      'T1': T1_fit,
      'C': C,
      'A_err': perr[0],
      'T1_err': perr[1],
      'C_err': perr[2],
      'pcov': pcov
  }


Save the T1 decay plot (data + fitted curve)

In [14]:
def save_t1_plot(delays, populations, fit_result, path=None):
  import os
  import matplotlib.pyplot as plt
  os.makedirs(os.path.dirname(path), exist_ok=True)
  t_fine = np.linspace(delays.min(), delays.max(), 300)
  fit_curve = _t1_decay_model(t_fine, fit_result["A"], fit_result["T1"], fit_result["C"])
  plt.figure(figsize=(6, 4))
  plt.plot(delays, populations, "o", label="measured", markersize=4)
  plt.plot(t_fine, fit_curve, "-",
              label=f"fit: T1 = {fit_result['T1']:.2f} \u00b1 {fit_result['T1_err']:.2f}")
  plt.xlabel("Delay time")
  plt.ylabel("Population in |1>")
  plt.title("T1 Relaxation Measurement")
  plt.legend()
  plt.tight_layout()
  plt.savefig(path, dpi=150)
  plt.close()
  return path

Now let us take a value for T1 and check the simulation

In [15]:
if __name__ == "__main__":
    print("T1 measurement experiment demo (standalone)")
    print("=" * 50)

    true_T1 = 50.0  # e.g. microseconds
    delays = np.linspace(0, 5 * true_T1, 40)

    L_relax = relaxation_operator(true_T1)
    H0 = build_hamiltonian(detuning=0.0)
    print(f"Relaxation operator L_relax (T1={true_T1}):\n{L_relax}\n")
    print(f"Hamiltonian used for T1 experiment (no coherent dynamics):\n{H0}\n")

    background_T2_demo = 60.0
    L_dephase_demo = dephasing_operator(true_T1, background_T2_demo)
    print(f"Optional background dephasing operator (T1={true_T1}, T2={background_T2_demo}):\n"
          f"{L_dephase_demo}\n")

    # Noiseless (exact expectation value) run
    data_ideal = run_t1_experiment(true_T1, delays)
    fit_ideal = fit_T1(data_ideal["delays"], data_ideal["pop_measured"])
    print(f"[Noiseless] True T1 = {true_T1}, Fitted T1 = {fit_ideal['T1']:.3f} "
          f"+/- {fit_ideal['T1_err']:.3f}")

    # Realistic run with finite-shot readout noise
    data_noisy = run_t1_experiment(true_T1, delays, n_shots=2000, seed=42)
    fit_noisy = fit_T1(data_noisy["delays"], data_noisy["pop_measured"])
    print(f"[2000 shots] True T1 = {true_T1}, Fitted T1 = {fit_noisy['T1']:.3f} "
          f"+/- {fit_noisy['T1_err']:.3f}")

    # Sanity check: background dephasing should not change fitted T1
    data_bg = run_t1_experiment(true_T1, delays, background_T2=60.0, n_shots=2000, seed=42)
    fit_bg = fit_T1(data_bg["delays"], data_bg["pop_measured"])
    print(f"[+background dephasing T2=60] Fitted T1 = {fit_bg['T1']:.3f} "
          f"+/- {fit_bg['T1_err']:.3f}  (should match the T1=50 case above)")



T1 measurement experiment demo (standalone)
Relaxation operator L_relax (T1=50.0):
Quantum object: dims=[[2], [2]], shape=(2, 2), type='oper', dtype=CSR, isherm=False
Qobj data =
[[0.         0.14142136]
 [0.         0.        ]]

Hamiltonian used for T1 experiment (no coherent dynamics):
Quantum object: dims=[[2], [2]], shape=(2, 2), type='oper', dtype=Dia, isherm=True
Qobj data =
[[0. 0.]
 [0. 0.]]

Optional background dephasing operator (T1=50.0, T2=60.0):
Quantum object: dims=[[2], [2]], shape=(2, 2), type='oper', dtype=CSR, isherm=True
Qobj data =
[[ 0.05773503  0.        ]
 [ 0.         -0.05773503]]

c_ops before mesolve: [Quantum object: dims=[[2], [2]], shape=(2, 2), type='oper', dtype=CSR, isherm=False
Qobj data =
[[0.         0.14142136]
 [0.         0.        ]]]
rho0 before mesolve:
Quantum object: dims=[[2], [2]], shape=(2, 2), type='oper', dtype=CSR, isherm=True
Qobj data =
[[0. 0.]
 [0. 1.]]
mesolve result.expect[0]: ([1.         0.87967292 0.77382438 0.68071231 0.59880